# MUStARD external evaluation notebook (v4-aligned)

This notebook performs **external validation on MUStARD** using the **final v4 in-domain checkpoint**.

It does the following, top to bottom:

1. authenticates Hugging Face if `HF_TOKEN` is available
2. locates the final v4 run folder and reads the saved validation thresholds
3. locates MUStARD files recursively under `D:\AI\datasets\MUStARD`
4. extracts utterance keyframes if needed
5. builds an external evaluation CSV from `sarcasm_data.json`
6. rebuilds the exact model architecture and loads `best_model.pt`
7. evaluates **multimodal** and **ensemble** outputs on MUStARD
8. saves synchronized JSON / CSV / TXT / PNG artifacts into a separate external-results folder

Run the notebook **top to bottom**.


In [1]:
import os
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Hugging Face login completed via HF_TOKEN environment variable.")
else:
    print("ℹ️ HF_TOKEN environment variable not found.")
    print("   If gated model access fails, run:")
    print('   from huggingface_hub import login')
    print('   login(token="YOUR_HF_READ_TOKEN", add_to_git_credential=False)')


ℹ️ HF_TOKEN environment variable not found.
   If gated model access fails, run:
   from huggingface_hub import login
   login(token="YOUR_HF_READ_TOKEN", add_to_git_credential=False)


In [2]:
import os

os.environ["HF_HOME"] = r"D:\AI\hf_cache"
os.environ["TRANSFORMERS_CACHE"] = r"D:\AI\hf_cache\transformers"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [3]:
import os
HF_TOKEN = os.environ.get("HF_TOKEN")

In [4]:
import sys, os, torch, transformers
print("PYTHON:", sys.executable)
print("TORCH :", torch.__version__)
print("TFM   :", transformers.__version__)
print("CUDA  :", torch.cuda.is_available())
print("HF_HOME:", os.environ.get("HF_HOME"))
print("TRANSFORMERS_CACHE:", os.environ.get("TRANSFORMERS_CACHE"))

PYTHON: D:\AI\envs\sarcasm\python.exe
TORCH : 2.11.0+cu128
TFM   : 5.6.1
CUDA  : True
HF_HOME: D:\AI\hf_cache
TRANSFORMERS_CACHE: D:\AI\hf_cache\transformers


In [5]:
import transformers, huggingface_hub
print(transformers.__version__)
print(huggingface_hub.__version__)
from transformers import AutoTokenizer, AutoModel
print("✅ transformers import OK")

5.6.1
1.11.0
✅ transformers import OK


In [6]:

import os, re, glob, json, math, random, shutil, time
from pathlib import Path

os.environ.setdefault("HF_HOME", r"D:\AI\hf_cache")
os.environ.setdefault("TRANSFORMERS_CACHE", r"D:\AI\hf_cache\transformers")
HF_TOKEN = os.environ.get("HF_TOKEN")
print("HF_TOKEN set:", HF_TOKEN is not None)

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from PIL import Image

from transformers import AutoTokenizer, AutoModel, CLIPVisionModel

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve, auc, average_precision_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ device:", device)


HF_TOKEN set: False
✅ device: cuda


In [7]:

# --- Locate final v4 in-domain run folder ---
DEFAULT_V4 = Path(r"D:\AI\outputs\runs_sarcasm_improve\improve_v4_lr3e-05_wd0.05_do0.5_bs8_ep8_fa0.6_fg1_SAMPLER")

if DEFAULT_V4.exists():
    V4_RUN_DIR = DEFAULT_V4
else:
    cand = list(Path(r"D:\AI\outputs\runs_sarcasm_improve").glob("*fa0.6_fg1_SAMPLER"))
    if not cand:
        raise FileNotFoundError("Could not locate the final v4 run folder under D:\\AI\\outputs\\runs_sarcasm_improve")
    V4_RUN_DIR = sorted(cand)[-1]

BEST_PT = V4_RUN_DIR / "best_model.pt"
BEST_METRICS = V4_RUN_DIR / "best_metrics.json"
V4_CONFIG = V4_RUN_DIR / "IMPROVE_V4_CONFIG.json"

for p in [BEST_PT, BEST_METRICS, V4_CONFIG]:
    if not p.exists():
        raise FileNotFoundError(f"Required v4 artifact not found: {p}")

best_metrics = json.loads(BEST_METRICS.read_text(encoding="utf-8"))
v4_cfg = json.loads(V4_CONFIG.read_text(encoding="utf-8"))

thr_multi_best = float(best_metrics["val_multi_thr"])
thr_ens_best = float(best_metrics["val_ens_thr"])
gwo_weights = np.array(best_metrics["gwo_weights"], dtype=float)

print("V4_RUN_DIR      =", V4_RUN_DIR)
print("thr_multi_best  =", thr_multi_best)
print("thr_ens_best    =", thr_ens_best)
print("gwo_weights     =", gwo_weights.tolist())
print("v4_cfg          =", json.dumps(v4_cfg, indent=2))


V4_RUN_DIR      = D:\AI\outputs\runs_sarcasm_improve\improve_v4_lr3e-05_wd0.05_do0.5_bs8_ep8_fa0.6_fg1_SAMPLER
thr_multi_best  = 0.525
thr_ens_best    = 0.05
gwo_weights     = [0.6406516563241764, 0.10851426883500451, 0.2507592046716443, 7.487016917479015e-05]
v4_cfg          = {
  "lr": 3e-05,
  "weight_decay": 0.05,
  "dropout": 0.5,
  "batch_size": 8,
  "epochs": 8,
  "focal_alpha": 0.6,
  "focal_gamma": 1.0,
  "sampler": true,
  "select_best_by": "macro_f1_tuned_multimodal"
}


In [8]:

# --- Locate Memotion dataset just to rebuild the same emoji vocabulary used by the checkpoint ---
MEMOTION_ROOT = Path(r"D:\AI\datasets\Memotion\memotion_dataset_7k")
if not MEMOTION_ROOT.exists():
    raise FileNotFoundError(f"Memotion root not found: {MEMOTION_ROOT}")

label_candidates = []
for pat in ["labels*.csv", "labels*.xlsx", "labels*.xls", "**/labels*.csv", "**/labels*.xlsx", "**/labels*.xls"]:
    label_candidates += glob.glob(str(MEMOTION_ROOT / pat), recursive=True)

label_candidates_sorted = sorted(label_candidates, key=lambda p: (0 if p.lower().endswith(".csv") else 1, len(p)))
CSV_PATH = label_candidates_sorted[0] if label_candidates_sorted else None
if CSV_PATH is None:
    raise FileNotFoundError(f"Could not find Memotion labels file under {MEMOTION_ROOT}")

memotion_df = pd.read_csv(CSV_PATH) if CSV_PATH.lower().endswith(".csv") else pd.read_excel(CSV_PATH)

def infer_col(cols, preferred_names, contains_names=()):
    cols_l = {c.lower(): c for c in cols}
    for name in preferred_names:
        if name.lower() in cols_l:
            return cols_l[name.lower()]
    for c in cols:
        cl = c.lower()
        if any(x in cl for x in contains_names):
            return c
    return None

TEXT_COL = infer_col(memotion_df.columns, ["text_corrected", "ocr_text", "text", "caption"], contains_names=["text", "caption", "ocr"])
LABEL_COL = infer_col(memotion_df.columns, ["sarcasm", "label"], contains_names=["sarcasm", "label"])
IMG_COL = infer_col(memotion_df.columns, ["image_name", "image", "img", "filename"], contains_names=["image", "img", "file"])

if TEXT_COL is None or LABEL_COL is None or IMG_COL is None:
    raise RuntimeError(f"Could not infer Memotion columns. Found columns: {list(memotion_df.columns)}")

def map_binary_sarcasm(x):
    s = str(x).strip().lower()
    if s in {"1", "true", "yes", "sarcastic", "very_twisted", "twisted_meaning", "general"}:
        return 1
    if s in {"0", "false", "no", "not_sarcastic", "not sarcastic"}:
        return 0
    try:
        return int(float(s) > 0)
    except Exception:
        return 1 if "sar" in s or "twist" in s else 0

memotion_df["binary_label"] = memotion_df[LABEL_COL].apply(map_binary_sarcasm)

EMOJI_RE = re.compile("["  
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U00002600-\U000026FF"
"]+", flags=re.UNICODE)

def extract_emojis(text: str):
    if not isinstance(text, str):
        return []
    return EMOJI_RE.findall(text)

emoji_counter = {}
for txt in memotion_df[TEXT_COL].astype(str).tolist():
    for e in extract_emojis(txt):
        emoji_counter[e] = emoji_counter.get(e, 0) + 1

emoji_items = sorted(emoji_counter.items(), key=lambda kv: (-kv[1], kv[0]))
emoji2id = {e: i+1 for i, (e, _) in enumerate(emoji_items)}

def emojis_to_ids(text, max_emojis=16):
    ids = [emoji2id.get(e, 0) for e in extract_emojis(text)[:max_emojis]]
    if len(ids) < max_emojis:
        ids += [0] * (max_emojis - len(ids))
    return ids

print("TEXT_COL:", TEXT_COL)
print("LABEL_COL:", LABEL_COL)
print("IMG_COL:", IMG_COL)
print("emoji vocab size:", len(emoji2id))


TEXT_COL: text_corrected
LABEL_COL: sarcasm
IMG_COL: image_name
emoji vocab size: 3


In [10]:

# --- Locate MUStARD files recursively and prepare external evaluation resources ---
MUSTARD_BASE = Path(r"D:\AI\datasets\MUStARD")
if not MUSTARD_BASE.exists():
    raise FileNotFoundError(f"MUStARD base folder not found: {MUSTARD_BASE}")

json_hits = list(MUSTARD_BASE.rglob("sarcasm_data.json"))
if not json_hits:
    raise FileNotFoundError(
        "Could not find sarcasm_data.json under D:\\AI\\datasets\\MUStARD.\n"
        "Please make sure the full MUStARD dataset/repo is present, not only the raw videos folder."
    )
ANN_PATH = json_hits[0]

utt_dirs = [p for p in MUSTARD_BASE.rglob("utterances_final") if p.is_dir()]
if not utt_dirs:
    raise FileNotFoundError(
        "Could not find utterances_final under D:\\AI\\datasets\\MUStARD.\n"
        "Expected something like ...\\raw_videos\\mmsd_raw_data\\utterances_final"
    )
UTT_DIR = utt_dirs[0]

KEYFRAME_DIR = MUSTARD_BASE / "_keyframes_utt_local"
KEYFRAME_DIR.mkdir(parents=True, exist_ok=True)

EVAL_DIR = MUSTARD_BASE / "_eval_v4_external"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

RESULT_DIR = EVAL_DIR / "results_v4_multimodal_primary"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("ANN_PATH     =", ANN_PATH)
print("UTT_DIR      =", UTT_DIR)
print("KEYFRAME_DIR =", KEYFRAME_DIR)
print("RESULT_DIR   =", RESULT_DIR)


ANN_PATH     = D:\AI\datasets\MUStARD\data\sarcasm_data.json
UTT_DIR      = D:\AI\datasets\MUStARD\mmsd_raw_data\utterances_final
KEYFRAME_DIR = D:\AI\datasets\MUStARD\_keyframes_utt_local
RESULT_DIR   = D:\AI\datasets\MUStARD\_eval_v4_external\results_v4_multimodal_primary


In [11]:

# --- Extract one keyframe per utterance video (middle frame) ---
import cv2

vids = sorted(list(UTT_DIR.glob("*.mp4")))
print("Utterance videos found:", len(vids))

fail = 0
written = 0
for vp in tqdm(vids, desc="Extract keyframes"):
    stem = vp.stem
    op = KEYFRAME_DIR / f"{stem}.jpg"
    if op.exists():
        continue

    cap = cv2.VideoCapture(str(vp))
    if not cap.isOpened():
        fail += 1
        continue

    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, n // 2))
    ok, frame = cap.read()
    cap.release()

    if ok and frame is not None:
        cv2.imwrite(str(op), frame)
        written += 1
    else:
        fail += 1

print("✅ keyframes available at:", KEYFRAME_DIR)
print("newly written:", written)
print("failed videos:", fail)


Utterance videos found: 690


Extract keyframes:   0%|          | 0/690 [00:00<?, ?it/s]

✅ keyframes available at: D:\AI\datasets\MUStARD\_keyframes_utt_local
newly written: 682
failed videos: 8


In [12]:

# --- Build MUStARD evaluation CSV ---
data = json.loads(Path(ANN_PATH).read_text(encoding="utf-8"))

rows = []
for clip_id, rec in data.items():
    utter = rec.get("utterance", "")
    ctx = rec.get("context", [])
    ctx_text = " ".join(ctx) if isinstance(ctx, list) else str(ctx)
    text = (ctx_text + " " + str(utter)).strip()
    y = int(bool(rec.get("sarcasm", False)))
    image_name = f"{clip_id}.jpg"
    img_path = KEYFRAME_DIR / image_name
    rows.append({
        "video_stem": str(clip_id),
        "text": text,
        "binary_label": y,
        "image_name": image_name,
        "image_exists": int(img_path.exists())
    })

mustard_df = pd.DataFrame(rows)
mustard_df = mustard_df[mustard_df["image_exists"] == 1].copy().reset_index(drop=True)

EVAL_CSV = EVAL_DIR / "mustard_eval_v4.csv"
mustard_df.to_csv(EVAL_CSV, index=False)

print(mustard_df.head(3))
print("Total usable samples:", len(mustard_df))
print("Positive fraction:", float(mustard_df["binary_label"].mean()) if len(mustard_df) else None)
print("Saved CSV:", EVAL_CSV)


  video_stem                                               text  binary_label  \
0       1_60  I never would have identified the fingerprints...             1   
1       1_70  This is one of my favorite places to kick back...             1   
2       1_80  Here we go. Pad thai, no peanuts. But does it ...             0   

  image_name  image_exists  
0   1_60.jpg             1  
1   1_70.jpg             1  
2   1_80.jpg             1  
Total usable samples: 682
Positive fraction: 0.49560117302052786
Saved CSV: D:\AI\datasets\MUStARD\_eval_v4_external\mustard_eval_v4.csv


In [13]:

TEXT_MODEL = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL,
    token=HF_TOKEN,
    use_fast=False,
)

resnet_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

clip_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466,0.4578275,0.40821073],
                         std=[0.26862954,0.26130258,0.27577711]),
])

class MemeDataset(Dataset):
    def __init__(self, df, images_dir, text_col, max_len=64, max_emojis=16):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.text_col = text_col
        self.max_len = max_len
        self.max_emojis = max_emojis

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, str(r["image_name"]))
        y = int(r["binary_label"])
        text = str(r[self.text_col])

        ok = 1
        try:
            img = Image.open(img_path).convert("RGB")
            pixel_resnet = resnet_transform(img)
            pixel_clip   = clip_transform(img)
        except Exception:
            pixel_resnet = torch.zeros(3,224,224)
            pixel_clip   = torch.zeros(3,224,224)
            ok = 0

        enc = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        eids = torch.tensor(emojis_to_ids(text, self.max_emojis), dtype=torch.long)

        return {
            "pixel_resnet": pixel_resnet,
            "pixel_clip": pixel_clip,
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "emoji_ids": eids,
            "label": torch.tensor(y, dtype=torch.float32),
            "ok": torch.tensor(ok, dtype=torch.long)
        }

def collate_fn(batch):
    batch = [b for b in batch if int(b["ok"]) == 1]
    if len(batch) == 0:
        return None
    out = {}
    for k in ["pixel_resnet","pixel_clip","input_ids","attention_mask","emoji_ids","label"]:
        out[k] = torch.stack([b[k] for b in batch], dim=0)
    return out

MAX_LEN = 64
mustard_ds = MemeDataset(mustard_df, str(KEYFRAME_DIR), "text", max_len=MAX_LEN)
mustard_loader = DataLoader(mustard_ds, batch_size=32, shuffle=False, collate_fn=collate_fn,
                            num_workers=0, pin_memory=True, persistent_workers=False)

print("✅ MUStARD dataset + loader ready:", len(mustard_ds))


✅ MUStARD dataset + loader ready: 682


In [14]:


import torch, torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights
from transformers import AutoModel, CLIPVisionModel

class SarcasmPaperModel(nn.Module):
    def __init__(self, emoji_vocab_size, dropout=0.3):
        super().__init__()

        TEXT_MODEL = "ai4bharat/indic-bert"
        try:
            self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL, token=HF_TOKEN, use_safetensors=True)
        except Exception:
            self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL, token=HF_TOKEN)

        self.resnet = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.resnet.fc = nn.Identity()        # 2048
        self.img_a_proj = nn.Linear(2048, 512)

        try:
            self.clip_vision = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32", use_safetensors=True)
        except Exception:
            self.clip_vision = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch32")

        self.img_b_proj = nn.Linear(self.clip_vision.config.hidden_size, 512)

        self.img_fuse = nn.Linear(1024, 768)

        self.emoji_emb = nn.Embedding(emoji_vocab_size + 1, 256, padding_idx=0)
        self.emoji_proj = nn.Linear(256, 768)

        self.attn_ti = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)
        self.attn_te = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)

        self.gate_ie = nn.Sequential(nn.Linear(768 + 768, 768), nn.Sigmoid())

        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(768)

       
        self.head_text  = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
        self.head_image = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
        self.head_emoji = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
        self.head_multi = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))

       
        self.sent_text  = nn.Linear(768, 1)
        self.sent_image = nn.Linear(768, 1)
        self.sent_emoji = nn.Linear(768, 1)

    def masked_emoji_mean(self, emoji_ids):
        emb = self.emoji_emb(emoji_ids)                   
        mask = (emoji_ids != 0).float().unsqueeze(-1)      
        denom = mask.sum(dim=1).clamp(min=1.0)
        pooled = (emb * mask).sum(dim=1) / denom
        return pooled

    def forward(self, pixel_resnet, pixel_clip, input_ids, attention_mask, emoji_ids):
        tout = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        Htext = tout.last_hidden_state
        htext = Htext[:, 0, :]

        fa2048 = self.resnet(pixel_resnet)
        fa = self.img_a_proj(fa2048)

        vout = self.clip_vision(pixel_values=pixel_clip)
        vb = vout.pooler_output if getattr(vout, "pooler_output", None) is not None else vout.last_hidden_state[:,0,:]
        fb = self.img_b_proj(vb)

        fimg = self.img_fuse(torch.cat([fa, fb], dim=-1))

        e = self.masked_emoji_mean(emoji_ids)
        femoji = self.emoji_proj(e)

        img_tok = fimg.unsqueeze(1)
        attn_ti, _ = self.attn_ti(Htext, img_tok, img_tok)
        H_ti = self.ln(Htext + self.dropout(attn_ti))

        emo_tok = femoji.unsqueeze(1)
        attn_te, _ = self.attn_te(H_ti, emo_tok, emo_tok)
        H_att = self.ln(H_ti + self.dropout(attn_te))
        h_att_cls = H_att[:, 0, :]

        g = self.gate_ie(torch.cat([fimg, femoji], dim=-1))
        fimg_g = g * fimg

        fused = self.ln(h_att_cls + fimg_g + femoji)

       
        logit_text  = self.head_text(htext).squeeze(-1)
        logit_image = self.head_image(fimg_g).squeeze(-1)
        logit_emoji = self.head_emoji(femoji).squeeze(-1)
        logit_multi = self.head_multi(fused).squeeze(-1)

       
        st_logit = self.sent_text(htext).squeeze(-1)
        si_logit = self.sent_image(fimg).squeeze(-1)
        se_logit = self.sent_emoji(femoji).squeeze(-1)

      
        st = torch.sigmoid(st_logit)
        si = torch.sigmoid(si_logit)
        se = torch.sigmoid(se_logit)
        incong = (torch.abs(st - si) + torch.abs(st - se) + torch.abs(si - se)) / 3.0

        return logit_text, logit_image, logit_emoji, logit_multi, incong, st_logit, si_logit, se_logit

model = SarcasmPaperModel(emoji_vocab_size=len(emoji2id), dropout=0.3).to(device)
print("✅ model ready (LOGITS + separate pixels)")

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.LayerNorm.weight     | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

✅ model ready (LOGITS + separate pixels)


In [15]:

def eval_probs(model, loader):
    model.eval()
    ys, p_multi_all, br_all = [], [], []
    for batch in loader:
        if batch is None:
            continue
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            logit_text, logit_image, logit_emoji, logit_multi, incong, st_logit, si_logit, se_logit = model(
                batch["pixel_resnet"], batch["pixel_clip"],
                batch["input_ids"], batch["attention_mask"], batch["emoji_ids"]
            )
        y = batch["label"].detach().float().cpu().numpy()
        ys.append(y)

        p_text = torch.sigmoid(logit_text).detach().cpu().numpy()
        p_image = torch.sigmoid(logit_image).detach().cpu().numpy()
        p_emoji = torch.sigmoid(logit_emoji).detach().cpu().numpy()
        p_multi = torch.sigmoid(logit_multi).detach().cpu().numpy()

        p_multi_all.append(p_multi)
        br_all.append(np.stack([p_text, p_image, p_emoji, p_multi], axis=1))

    y_true = np.concatenate(ys) if ys else np.array([])
    p_multi = np.concatenate(p_multi_all) if p_multi_all else np.array([])
    br = np.concatenate(br_all) if br_all else np.zeros((0, 4))
    return y_true, p_multi, br

def metrics_from_probs(y_true, p, thr=0.5):
    if len(y_true) == 0:
        return {"acc": 0.0, "macro_f1": 0.0, "auc": 0.0}
    yhat = (p >= thr).astype(int)
    return {
        "acc": float(accuracy_score(y_true, yhat)),
        "macro_f1": float(f1_score(y_true, yhat, average="macro", zero_division=0)),
        "auc": float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) == 2 else 0.0
    }

print("✅ eval helpers ready")


✅ eval helpers ready


In [16]:

# --- Load final v4 checkpoint ---
model = SarcasmPaperModel(emoji_vocab_size=len(emoji2id), dropout=float(v4_cfg["dropout"])).to(device)
state = torch.load(BEST_PT, map_location=device)
model.load_state_dict(state, strict=True)
model.eval()
print("✅ Loaded checkpoint:", BEST_PT)


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     |  | 
---------------------------------+------------+--+-
predictions.LayerNorm.weight     | UNEXPECTED |  | 
predictions.dense.bias           | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
predictions.bias                 | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 
predictions.dense.weight         | UNEXPECTED |  | 
predictions.LayerNorm.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

✅ Loaded checkpoint: D:\AI\outputs\runs_sarcasm_improve\improve_v4_lr3e-05_wd0.05_do0.5_bs8_ep8_fa0.6_fg1_SAMPLER\best_model.pt


In [17]:

# --- Run synchronized MUStARD evaluation ---
yt, p_multi, br = eval_probs(model, mustard_loader)

if len(yt) == 0:
    raise RuntimeError("MUStARD loader produced zero usable samples.")

p_ens = (br * gwo_weights.reshape(1, 4)).sum(axis=1)

multi_0_5 = metrics_from_probs(yt, p_multi, thr=0.5)
multi_best = metrics_from_probs(yt, p_multi, thr=thr_multi_best)
ens_0_5 = metrics_from_probs(yt, p_ens, thr=0.5)
ens_best = metrics_from_probs(yt, p_ens, thr=thr_ens_best)

yhat_multi_0_5 = (p_multi >= 0.5).astype(int)
yhat_multi_best = (p_multi >= thr_multi_best).astype(int)
yhat_ens_0_5 = (p_ens >= 0.5).astype(int)
yhat_ens_best = (p_ens >= thr_ens_best).astype(int)

pred_df = mustard_df.copy()
pred_df["p_multi"] = p_multi
pred_df["p_ens"] = p_ens
pred_df["yhat_multi_0_5"] = yhat_multi_0_5
pred_df["yhat_multi_best"] = yhat_multi_best
pred_df["yhat_ens_0_5"] = yhat_ens_0_5
pred_df["yhat_ens_best"] = yhat_ens_best

pred_csv = RESULT_DIR / "mustard_predictions_v4.csv"
pred_df.to_csv(pred_csv, index=False)

final_eval = {
    "source_run_dir": str(V4_RUN_DIR),
    "mustard_eval_csv": str(EVAL_CSV),
    "test_size": int(len(yt)),
    "test_pos_frac": float(np.mean(yt)),
    "thresholds": {
        "val_multi_thr": float(thr_multi_best),
        "val_ens_thr": float(thr_ens_best)
    },
    "multi_thr_0_5": {**multi_0_5, "threshold": 0.5, "positive_rate": float(np.mean(yhat_multi_0_5))},
    "multi_thr_best": {**multi_best, "threshold": float(thr_multi_best), "positive_rate": float(np.mean(yhat_multi_best))},
    "ens_thr_0_5": {**ens_0_5, "threshold": 0.5, "positive_rate": float(np.mean(yhat_ens_0_5))},
    "ens_thr_best": {**ens_best, "threshold": float(thr_ens_best), "positive_rate": float(np.mean(yhat_ens_best))},
    "ensemble_weights": gwo_weights.tolist()
}

(final_eval_path := RESULT_DIR / "final_eval_mustard_v4.json").write_text(json.dumps(final_eval, indent=2), encoding="utf-8")

(RESULT_DIR / "classification_report_mustard_multi_best.txt").write_text(
    classification_report(yt, yhat_multi_best, digits=4, zero_division=0),
    encoding="utf-8"
)
(RESULT_DIR / "classification_report_mustard_ens_best.txt").write_text(
    classification_report(yt, yhat_ens_best, digits=4, zero_division=0),
    encoding="utf-8"
)

summary = pd.DataFrame([
    {"dataset":"MUStARD", "model":"multimodal", "operating_point":"thr_0_5", **multi_0_5, "threshold":0.5},
    {"dataset":"MUStARD", "model":"multimodal", "operating_point":"thr_best", **multi_best, "threshold":float(thr_multi_best)},
    {"dataset":"MUStARD", "model":"ensemble", "operating_point":"thr_0_5", **ens_0_5, "threshold":0.5},
    {"dataset":"MUStARD", "model":"ensemble", "operating_point":"thr_best", **ens_best, "threshold":float(thr_ens_best)},
])
summary.to_csv(RESULT_DIR / "table6_mustard_reprofix.csv", index=False)

print(json.dumps(final_eval, indent=2))
print("✅ Saved:", pred_csv)
print("✅ Saved:", final_eval_path)


{
  "source_run_dir": "D:\\AI\\outputs\\runs_sarcasm_improve\\improve_v4_lr3e-05_wd0.05_do0.5_bs8_ep8_fa0.6_fg1_SAMPLER",
  "mustard_eval_csv": "D:\\AI\\datasets\\MUStARD\\_eval_v4_external\\mustard_eval_v4.csv",
  "test_size": 682,
  "test_pos_frac": 0.49560117721557617,
  "thresholds": {
    "val_multi_thr": 0.525,
    "val_ens_thr": 0.05
  },
  "multi_thr_0_5": {
    "acc": 0.47653958944281527,
    "macro_f1": 0.4456458630747058,
    "auc": 0.46804905738268887,
    "threshold": 0.5,
    "positive_rate": 0.7404692082111437
  },
  "multi_thr_best": {
    "acc": 0.48240469208211145,
    "macro_f1": 0.47901640124735173,
    "auc": 0.46804905738268887,
    "threshold": 0.525,
    "positive_rate": 0.5850439882697948
  },
  "ens_thr_0_5": {
    "acc": 0.5043988269794721,
    "macro_f1": 0.33528265107212474,
    "auc": 0.4608074170909591,
    "threshold": 0.5,
    "positive_rate": 0.0
  },
  "ens_thr_best": {
    "acc": 0.49560117302052786,
    "macro_f1": 0.33137254901960783,
    "auc": 0.

In [19]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score

In [20]:

# --- Plot and save MUStARD figures ---
def save_cm(y_true, y_hat, title, out_path):
    cm = confusion_matrix(y_true, y_hat)
    fig, ax = plt.subplots(figsize=(5,4), dpi=200)
    im = ax.imshow(cm, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("Pred")
    ax.set_ylabel("True")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i,j] > cm.max()/2 else "black")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Count")
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

def save_roc(y_true, p, title, out_path):
    fpr, tpr, _ = roc_curve(y_true, p)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(5,4), dpi=200)
    ax.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    ax.plot([0,1], [0,1], "--", linewidth=1)
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

def save_pr(y_true, p, title, out_path):
    prec, rec, _ = precision_recall_curve(y_true, p)
    pr_auc = average_precision_score(y_true, p)
    fig, ax = plt.subplots(figsize=(5,4), dpi=200)
    ax.plot(rec, prec, label=f"PR-AUC={pr_auc:.3f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)

save_cm(yt, yhat_multi_best, "MUStARD Confusion Matrix (multimodal, tuned threshold)", RESULT_DIR / "fig6_confusion_matrix_multimodal.png")
save_roc(yt, p_multi, "MUStARD ROC (multimodal)", RESULT_DIR / "fig6_roc_multimodal.png")
save_pr(yt, p_multi, "MUStARD PR (multimodal)", RESULT_DIR / "fig6_pr_multimodal.png")

save_cm(yt, yhat_ens_best, "MUStARD Confusion Matrix (ensemble, tuned threshold)", RESULT_DIR / "mustard_confusion_matrix_ensemble.png")
save_roc(yt, p_ens, "MUStARD ROC (ensemble)", RESULT_DIR / "mustard_roc_ensemble.png")
save_pr(yt, p_ens, "MUStARD PR (ensemble)", RESULT_DIR / "mustard_pr_ensemble.png")

manifest = {
    "source_run_dir": str(V4_RUN_DIR),
    "result_dir": str(RESULT_DIR),
    "files": sorted([p.name for p in RESULT_DIR.iterdir() if p.is_file()])
}
(RESULT_DIR / "artifact_manifest_mustard_v4.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("✅ MUStARD figures + manifest saved to:", RESULT_DIR)
print(json.dumps(manifest, indent=2))


✅ MUStARD figures + manifest saved to: D:\AI\datasets\MUStARD\_eval_v4_external\results_v4_multimodal_primary
{
  "source_run_dir": "D:\\AI\\outputs\\runs_sarcasm_improve\\improve_v4_lr3e-05_wd0.05_do0.5_bs8_ep8_fa0.6_fg1_SAMPLER",
  "result_dir": "D:\\AI\\datasets\\MUStARD\\_eval_v4_external\\results_v4_multimodal_primary",
  "files": [
    "classification_report_mustard_ens_best.txt",
    "classification_report_mustard_multi_best.txt",
    "fig6_confusion_matrix_multimodal.png",
    "fig6_pr_multimodal.png",
    "fig6_roc_multimodal.png",
    "final_eval_mustard_v4.json",
    "mustard_confusion_matrix_ensemble.png",
    "mustard_pr_ensemble.png",
    "mustard_predictions_v4.csv",
    "mustard_roc_ensemble.png",
    "table6_mustard_reprofix.csv"
  ]
}
